### Imports

In [24]:
import pandas as pd
import json
import pickle

### Data Overview

In [56]:
# Live data, not enriched with historical phishing data
df_live = pd.read_json("certs_fallback_live.jsonl", lines=True)
df_live.head()

,schema_version,timestamp,cert_index,fingerprint,serial,not_before,not_after,domains,subject,issuer,source
0,1,2026-04-28 16:15:10.369275+00:00,2773474520,1A:51:1F:CA:C2:89:58:79:97:EF:09:76:7B:3B:C4:0...,4.254407e+37,1777389189,1777478619,[161725127-gclb-alpn.ccm-probers-prod.certsbri...,{'CN': '161725127-gclb-alpn.ccm-probers-prod.c...,"{'CN': 'WR3', 'O': 'Google Trust Services', 'C...","{'name': 'Google 'Argon2026h1' log', 'url': 'h..."
1,1,2026-04-28 16:15:10.370322+00:00,2773474521,A4:64:8A:15:FD:8C:A4:F4:7D:82:34:05:6E:2E:F5:6...,2.175468e+36,1777392788,1777479129,"[eap.ameren.com, imperva.com]","{'CN': 'imperva.com', 'O': None, 'C': None}",{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,"{'name': 'Google 'Argon2026h1' log', 'url': 'h..."
2,1,2026-04-28 16:15:10.370588+00:00,2773474522,B9:EA:BA:5F:58:27:FD:E0:D1:04:5C:06:91:6F:46:7...,2.511507e+36,1777392788,1777479129,"[eap.qa.ameren.com, imperva.com]","{'CN': 'imperva.com', 'O': None, 'C': None}",{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,"{'name': 'Google 'Argon2026h1' log', 'url': 'h..."
3,1,2026-04-28 16:15:10.370831+00:00,2773474523,0B:95:84:CB:15:63:0C:D7:BB:C0:71:96:0E:45:DC:5...,1.346110e+36,1777392788,1777479129,"[ihu-qa.ameren.com, imperva.com]","{'CN': 'imperva.com', 'O': None, 'C': None}",{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,"{'name': 'Google 'Argon2026h1' log', 'url': 'h..."
4,1,2026-04-28 16:15:10.371371+00:00,2773474527,DB:9D:C8:22:F7:87:31:83:76:2B:DD:8A:B5:F6:17:E...,2.468586e+38,1777389177,1777475576,[basic-check.http-01.production.haplorrhini.com],{'CN': 'basic-check.http-01.production.haplorr...,"{'CN': 'WR3', 'O': 'Google Trust Services', 'C...","{'name': 'Google 'Argon2026h1' log', 'url': 'h..."


In [57]:
df_live_exploded = df_live.explode("domains").rename(columns={"domains": "domain"})
df_exploded = df_live_exploded[df_live_exploded["domain"].notna()].reset_index(drop=True)

In [58]:
df_live_exploded.head()

,schema_version,timestamp,cert_index,fingerprint,serial,not_before,not_after,domain,subject,issuer,source
0,1,2026-04-28 16:15:10.369275+00:00,2773474520,1A:51:1F:CA:C2:89:58:79:97:EF:09:76:7B:3B:C4:0...,4.254407e+37,1777389189,1777478619,161725127-gclb-alpn.ccm-probers-prod.certsbrid...,{'CN': '161725127-gclb-alpn.ccm-probers-prod.c...,"{'CN': 'WR3', 'O': 'Google Trust Services', 'C...","{'name': 'Google 'Argon2026h1' log', 'url': 'h..."
1,1,2026-04-28 16:15:10.370322+00:00,2773474521,A4:64:8A:15:FD:8C:A4:F4:7D:82:34:05:6E:2E:F5:6...,2.175468e+36,1777392788,1777479129,eap.ameren.com,"{'CN': 'imperva.com', 'O': None, 'C': None}",{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,"{'name': 'Google 'Argon2026h1' log', 'url': 'h..."
1,1,2026-04-28 16:15:10.370322+00:00,2773474521,A4:64:8A:15:FD:8C:A4:F4:7D:82:34:05:6E:2E:F5:6...,2.175468e+36,1777392788,1777479129,imperva.com,"{'CN': 'imperva.com', 'O': None, 'C': None}",{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,"{'name': 'Google 'Argon2026h1' log', 'url': 'h..."
2,1,2026-04-28 16:15:10.370588+00:00,2773474522,B9:EA:BA:5F:58:27:FD:E0:D1:04:5C:06:91:6F:46:7...,2.511507e+36,1777392788,1777479129,eap.qa.ameren.com,"{'CN': 'imperva.com', 'O': None, 'C': None}",{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,"{'name': 'Google 'Argon2026h1' log', 'url': 'h..."
2,1,2026-04-28 16:15:10.370588+00:00,2773474522,B9:EA:BA:5F:58:27:FD:E0:D1:04:5C:06:91:6F:46:7...,2.511507e+36,1777392788,1777479129,imperva.com,"{'CN': 'imperva.com', 'O': None, 'C': None}",{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,"{'name': 'Google 'Argon2026h1' log', 'url': 'h..."


In [59]:
df_live_exploded.columns

Index(['schema_version', 'timestamp', 'cert_index', 'fingerprint', 'serial',
       'not_before', 'not_after', 'domain', 'subject', 'issuer', 'source'],
      dtype='str')

### Add Labels

In [ ]:
df_live_labels   = pd.read_json("phishtank_labels_live.jsonl", lines=True)
df_live_certs = pd.read_json("certs_fallback_live.jsonl", lines=True)

df = (df_live_certs
      .explode("domains")
      .rename(columns={"domains": "domain"})
      .dropna(subset=["domain"]))

df["domain"] = df["domain"].str.lower().str.lstrip("*.")

df = df.merge(df_live_labels[["domain", "y", "label_source", "label_ts"]], on="domain", how="left")
df["y"] = df["y"].fillna(0).astype(int)
df.head()

,schema_version,timestamp,cert_index,fingerprint,serial,not_before,not_after,domain,subject,issuer,source,y,label_source,label_ts
0,1,2026-04-28 16:15:10.369275+00:00,2773474520,1A:51:1F:CA:C2:89:58:79:97:EF:09:76:7B:3B:C4:0...,4.254407e+37,1777389189,1777478619,161725127-gclb-alpn.ccm-probers-prod.certsbrid...,{'CN': '161725127-gclb-alpn.ccm-probers-prod.c...,"{'CN': 'WR3', 'O': 'Google Trust Services', 'C...","{'name': 'Google 'Argon2026h1' log', 'url': 'h...",0,NaN,NaN
1,1,2026-04-28 16:15:10.370322+00:00,2773474521,A4:64:8A:15:FD:8C:A4:F4:7D:82:34:05:6E:2E:F5:6...,2.175468e+36,1777392788,1777479129,eap.ameren.com,"{'CN': 'imperva.com', 'O': None, 'C': None}",{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,"{'name': 'Google 'Argon2026h1' log', 'url': 'h...",0,NaN,NaN
2,1,2026-04-28 16:15:10.370322+00:00,2773474521,A4:64:8A:15:FD:8C:A4:F4:7D:82:34:05:6E:2E:F5:6...,2.175468e+36,1777392788,1777479129,imperva.com,"{'CN': 'imperva.com', 'O': None, 'C': None}",{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,"{'name': 'Google 'Argon2026h1' log', 'url': 'h...",0,NaN,NaN
3,1,2026-04-28 16:15:10.370588+00:00,2773474522,B9:EA:BA:5F:58:27:FD:E0:D1:04:5C:06:91:6F:46:7...,2.511507e+36,1777392788,1777479129,eap.qa.ameren.com,"{'CN': 'imperva.com', 'O': None, 'C': None}",{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,"{'name': 'Google 'Argon2026h1' log', 'url': 'h...",0,NaN,NaN
4,1,2026-04-28 16:15:10.370588+00:00,2773474522,B9:EA:BA:5F:58:27:FD:E0:D1:04:5C:06:91:6F:46:7...,2.511507e+36,1777392788,1777479129,imperva.com,"{'CN': 'imperva.com', 'O': None, 'C': None}",{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,"{'name': 'Google 'Argon2026h1' log', 'url': 'h...",0,NaN,NaN


## EDA

### Functions

In [30]:
def explode_domains(df):
    d = (df.explode("domains")
           .rename(columns={"domains": "domain"})
           .dropna(subset=["domain"]))
    d["domain"] = d["domain"].str.lower().str.strip().str.lstrip("*.")
    d = d[d["domain"].str.len() > 0]
    return d.reset_index(drop=True)

### Basic Data Review

In [53]:
df_live.describe()

,schema_version,cert_index,serial,not_before,not_after
count,13052.0,1.305200e+04,1.305200e+04,1.305200e+04,1.305200e+04
mean,1.0,2.773496e+09,2.830728e+45,1.775710e+09,1.778929e+09
std,0.0,9.307087e+03,3.629410e+46,5.404205e+06,1.554974e+06
min,1.0,2.773475e+09,3.284799e+16,1.736035e+09,1.767658e+09
25%,1.0,2.773493e+09,3.764075e+37,1.777334e+09,1.777483e+09
50%,1.0,2.773499e+09,1.862558e+38,1.777394e+09,1.777997e+09
75%,1.0,2.773504e+09,4.570842e+41,1.777396e+09,1.780013e+09
max,1.0,2.773508e+09,7.045454e+47,1.781784e+09,1.782864e+09


In [54]:
# Live CT log sample, assigning source to differentiate from historical phishing data
# df_live = pd.read_json("certs_fallback.jsonl", lines=True)
df_live["data_source"] = "ct_live"

# Historical phishing certs (already labeled y=1)
df_hist = pd.read_json("certs_phishtank_historical.jsonl", lines=True)
df_hist["data_source"] = "crtsh_historical"

# Labels for the live data
df_live_labels = pd.read_json("phishtank_labels.jsonl", lines=True)

print(f"Live certs:       {len(df_live):,}")
print(f"Historical certs: {len(df_hist):,}")
print(f"PhishTank labels: {len(df_live_labels):,}")

Live certs:       13,052
Historical certs: 573
PhishTank labels: 27,168


### Combining Historical Phishing and Latest Live Data (DO NOT USE FOR TIME SERIES ANALYSIS)

In [33]:
df_live_d = explode_domains(df_live)
df_hist_d = explode_domains(df_hist)

# Label live data via PhishTank join
df_live_d = df_live_d.merge(
    labels[["domain", "y", "label_source"]],
    on="domain", how="left"
)
df_live_d["y"] = df_live_d["y"].fillna(0).astype(int)

# Combine — historical is pre-labeled
df = pd.concat([df_live_d, df_hist_d], ignore_index=True)
df["y"] = df["y"].fillna(1).astype(int)  # hist records are all phishing

print(f"\nCombined domain rows: {len(df):,}")
print(df["y"].value_counts().rename({0: "legitimate/unknown", 1: "phishing"}))
print(f"\nPhishing rate: {df['y'].mean():.4%}")


Combined domain rows: 30,767
y
legitimate/unknown    29770
phishing                997
Name: count, dtype: int64

Phishing rate: 3.2405%


In [45]:
df.describe()

,schema_version,cert_index,not_before,not_after,y
count,30767.0,2.977000e+04,2.977000e+04,2.977000e+04,30767.000000
mean,1.0,2.773496e+09,1.775996e+09,1.778760e+09,0.032405
std,0.0,9.397328e+03,5.178584e+06,1.429536e+06,0.177076
min,1.0,2.773475e+09,1.736035e+09,1.767658e+09,0.000000
25%,1.0,2.773493e+09,1.777390e+09,1.777970e+09,0.000000
50%,1.0,2.773498e+09,1.777394e+09,1.778003e+09,0.000000
75%,1.0,2.773504e+09,1.777396e+09,1.779990e+09,0.000000
max,1.0,2.773508e+09,1.781784e+09,1.782864e+09,1.000000


In [47]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30767 entries, 0 to 30766
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   schema_version   30767 non-null  int64              
 1   timestamp        30767 non-null  datetime64[us, UTC]
 2   cert_index       29770 non-null  float64            
 3   fingerprint      29770 non-null  object             
 4   serial           30767 non-null  object             
 5   not_before       29770 non-null  float64            
 6   not_after        29770 non-null  float64            
 7   domain           30767 non-null  str                
 8   subject          30767 non-null  object             
 9   issuer           30767 non-null  object             
 10  source           30767 non-null  object             
 11  data_source      30767 non-null  str                
 12  y                30767 non-null  int64              
 13  label_source     997 non-nu

In [46]:
print("\n── Sample phishing rows ─────────────────────────────────")
df[df["y"] == 1][["domain", "issuer", "not_before", "not_after"]].head(5)


── Sample phishing rows ─────────────────────────────────


,domain,issuer,not_before,not_after
29770,www.dropbox.com--login.info,"{'CN': 'R3', 'O': 'Let's Encrypt', 'C': 'US', ...",NaN,NaN
29771,www.dropbox.com--login.info,"{'CN': 'R3', 'O': 'Let's Encrypt', 'C': 'US', ...",NaN,NaN
29772,www.dropbox.com--login.info,"{'CN': 'R3', 'O': 'Let's Encrypt', 'C': 'US', ...",NaN,NaN
29773,www.dropbox.com--login.info,"{'CN': 'R3', 'O': 'Let's Encrypt', 'C': 'US', ...",NaN,NaN
29774,www.dropbox.com--login.info,"{'CN': 'R3', 'O': 'Let's Encrypt', 'C': 'US', ...",NaN,NaN


In [43]:
print("\n── Sample legitimate rows ───────────────────────────────")
df[df["y"] == 0][["domain", "issuer", "not_before", "not_after"]].head(5)


── Sample legitimate rows ───────────────────────────────


,domain,issuer,not_before,not_after
0,161725127-gclb-alpn.ccm-probers-prod.certsbrid...,"{'CN': 'WR3', 'O': 'Google Trust Services', 'C...",1.777389e+09,1.777479e+09
1,eap.ameren.com,{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,1.777393e+09,1.777479e+09
2,imperva.com,{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,1.777393e+09,1.777479e+09
3,eap.qa.ameren.com,{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,1.777393e+09,1.777479e+09
4,imperva.com,{'CN': 'GlobalSign Atlas R3 DV TLS CA 2026 Q2'...,1.777393e+09,1.777479e+09


## Basic Feature Engineering

### Functions

In [35]:
import math
from collections import Counter

def entropy(s):
    s = s.replace(".", "")
    counts = Counter(s)
    total = len(s)
    return -sum((c/total) * math.log2(c/total) for c in counts.values()) if total else 0

def subdomain_count(domain):
    return len(domain.split(".")) - 2  # subtract SLD + TLD

def tld(domain):
    return domain.rsplit(".", 1)[-1].lower()

def issuer_org(row):
    return row.get("issuer", {}).get("O", None)

df_exploded["entropy"]         = df_exploded["domain"].apply(entropy)
df_exploded["subdomain_count"] = df_exploded["domain"].apply(subdomain_count)
df_exploded["tld"]             = df_exploded["domain"].apply(tld)
df_exploded["issuer_org"]      = df_exploded["issuer"].apply(issuer_org)
df_exploded["is_letsencrypt"]  = df_exploded["issuer_org"].str.contains("Let's Encrypt", na=False)

In [36]:
# pip install python-Levenshtein
from Levenshtein import distance

BRANDS = ["paypal", "amazon", "apple", "microsoft", "google", "facebook", "netflix", "instagram"]

def min_brand_distance(domain):
    # strip subdomains, compare registrable domain only
    base = domain.split(".")[-2] if len(domain.split(".")) >= 2 else domain
    return min(distance(base, brand) for brand in BRANDS)

df_exploded["brand_distance"] = df_exploded["domain"].apply(min_brand_distance)

In [37]:
import json
import math
import warnings
from collections import Counter
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from Levenshtein import distance as levenshtein
import tldextract

### Engineering